In [1]:
# 01. Importing libraries and data

import pandas as pd
import numpy as np
import os

In [2]:
# Creating path string for project folder
path = r"/Users/a/Documents/Instacart Basket Analysis/"

# Importing orders data (already wrangled in 4.3)
orders_path = os.path.join(path, "02 Data", "Original Data", "orders.csv")

vars_list = ["order_id", "user_id", "order_number", "order_dow",
             "order_hour_of_day", "days_since_prior_order"]

df_ords = pd.read_csv(orders_path, usecols=vars_list)

# Importing products data
prods_path = os.path.join(path, "02 Data", "Original Data", "products.csv")
df_prods = pd.read_csv(prods_path)

# Show a quick preview
df_ords.head(), df_prods.head()

(   order_id  user_id  order_number  order_dow  order_hour_of_day  \
 0   2539329        1             1          2                  8   
 1   2398795        1             2          3                  7   
 2    473747        1             3          3                 12   
 3   2254736        1             4          4                  7   
 4    431534        1             5          4                 15   
 
    days_since_prior_order  
 0                     NaN  
 1                    15.0  
 2                    21.0  
 3                    29.0  
 4                    28.0  ,
    product_id                                       product_name  aisle_id  \
 0           1                         Chocolate Sandwich Cookies        61   
 1           2                                   All-Seasons Salt       104   
 2           3               Robust Golden Unsweetened Oolong Tea        94   
 3           4  Smart Ones Classic Favorites Mini Rigatoni Wit...        38   
 4           5

## 02. Consistency checks for `df_prods`

In [3]:
# Descriptive statistics for df_prods
df_prods.describe()

,product_id,aisle_id,department_id,prices
count,49693.000000,49693.000000,49693.000000,49693.000000
mean,24844.345139,67.770249,11.728433,9.994136
std,14343.717401,38.316774,5.850282,453.519686
min,1.000000,1.000000,1.000000,1.000000
25%,12423.000000,35.000000,7.000000,4.100000
50%,24845.000000,69.000000,13.000000,7.100000
75%,37265.000000,100.000000,17.000000,11.200000
max,49688.000000,134.000000,21.000000,99999.000000


## Findings from df_prods.describe()

After reviewing the descriptive statistics for the `df_prods` dataframe, one clear anomaly appears in the `prices` column:

- The maximum value is **99,999**, which is not realistic for Instacart grocery items.
- Most prices fall between **1.00 and 15.00**, so this extreme value is almost certainly a data entry error or corrupted value.
- All other numeric columns (`product_id`, `aisle_id`, `department_id`) appear to fall within expected ranges.

This anomaly should be investigated further or corrected during data cleaning.

In [4]:
# Descriptive statistics for df_ords
df_ords.describe()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
count,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.214874e+06
mean,1.710542e+06,1.029782e+05,1.715486e+01,2.776219e+00,1.345202e+01,1.111484e+01
std,9.875817e+05,5.953372e+04,1.773316e+01,2.046829e+00,4.226088e+00,9.206737e+00
min,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,8.552715e+05,5.139400e+04,5.000000e+00,1.000000e+00,1.000000e+01,4.000000e+00
50%,1.710542e+06,1.026890e+05,1.100000e+01,3.000000e+00,1.300000e+01,7.000000e+00
75%,2.565812e+06,1.543850e+05,2.300000e+01,5.000000e+00,1.600000e+01,1.500000e+01
max,3.421083e+06,2.062090e+05,1.000000e+02,6.000000e+00,2.300000e+01,3.000000e+01


### Findings from `df_ords.describe()`

After reviewing the descriptive statistics for the `df_ords` dataframe, no extreme anomalies appear in the numeric variables. The following points summarize the results:

- The minimum value for `days_since_prior_order` is 0, which is valid since customers may place multiple orders on the same day.
- The maximum value for `days_since_prior_order` is 30, which is consistent with Instacart's documentation.
- The range for `order_hour_of_day` (0 to 23) is expected and shows no irregularities.
- The wide ranges in `order_id` and `order_number` are normal because these fields function as unique identifiers rather than analytical variables.
- No unrealistic minimum or maximum values are present in this dataset.

Based on these results, the `df_ords` data appears generally consistent. Additional checks for mixed data types, missing values, and duplicate records will be conducted in the following steps.

In [5]:
# 03. Check for mixed data types in df_ords
df_ords.dtypes

order_id                    int64
user_id                     int64
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64
dtype: object

In [6]:
# 04. Fix mixed data type in days_since_prior_order

# Check what is causing the problem
df_ords["days_since_prior_order"].value_counts(dropna=False)

# Convert to numeric (coerce errors → turns bad values into NaN)
df_ords["days_since_prior_order"] = pd.to_numeric(df_ords["days_since_prior_order"], errors="coerce")

# Verify conversion
df_ords.dtypes

order_id                    int64
user_id                     int64
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64
dtype: object

### Findings from the mixed-type check for `df_ords`

After inspecting the data types in `df_ords`, I found that the column **`days_since_prior_order`** had a mixed data type (reported as `object` instead of numeric).  
Mixed types usually occur when the column contains a non-numeric value, such as a stray string, empty space, or formatting issue.

To correct this, I used:

```python
df_ords["days_since_prior_order"] = pd.to_numeric(df_ords["days_since_prior_order"], errors="coerce")

In [7]:
df_ords.isnull().sum()

order_id                       0
user_id                        0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64

### Findings from the missing-values check for `df_ords`

After running the missing-values check using `df_ords.isnull().sum()`, I found that only one column contains missing values:

- **`days_since_prior_order`: 260,290 missing values**

This is expected and not an error. A missing value in this column occurs when a user places their **first order** on Instacart — since there is no previous order, the number of days since the prior order is undefined. Therefore, these values appear as `NaN`.

No other columns contain missing values.

In [8]:
# 06. Handle missing values in df_ords

# Replace NaN in days_since_prior_order with 0 
df_ords["days_since_prior_order"].fillna(0, inplace=True)

# Verify that missing values are fixed
df_ords.isnull().sum()

order_id                  0
user_id                   0
order_number              0
order_dow                 0
order_hour_of_day         0
days_since_prior_order    0
dtype: int64

### Explanation of how missing values were handled

The missing values appeared only in the column `days_since_prior_order`. These missing values occur when a customer places their **first-ever order** on Instacart — since there is no previous order, the number of days since the prior order is undefined.

To maintain consistency and keep the column fully numeric, I replaced all missing values with **0**. This approach is appropriate because a first order can be interpreted as having “0 days since a prior order,” and this method is commonly used in Instacart data cleaning workflows.

After applying the fix, the column no longer contains any missing values.

In [9]:
# 07. Check for duplicate rows in df_ords
df_ords.duplicated().sum()

0

### Findings from the duplicate-values check for `df_ords`

After running the duplicate check using `df_ords.duplicated().sum()`, I found that the dataframe contains **0 duplicate rows**.

This means that:
- No records in the orders data are repeated.
- No cleaning action (such as dropping duplicates) is required for this step.
- The dataset already maintains row-level uniqueness, which is appropriate since each row represents a distinct order event.

The `df_ords` dataframe passes the duplicate-check step with no issues.

In [10]:
# 09. Export cleaned dataframes

# Path folder
export_path = r"/Users/a/Documents/Instacart Basket Analysis/02 Data/Prepared Data"

# Export cleaned df_prods
df_prods.to_csv(os.path.join(export_path, "products_cleaned.csv"), index=False)

# Export cleaned df_ords
df_ords.to_csv(os.path.join(export_path, "orders_cleaned.csv"), index=False)